# ETH & ERC20 Transaction Data Fetcher

**Part of:** Ethereum Topological Anomaly Detection (ETH-TAD)  
**Paper:** Ofori-Boateng et al. (2021) - arXiv:2106.01806

## Description
Downloads both ETH native transactions and ERC20 token transfers from Xatu ClickHouse database,
then aggregates them into weekly parquet files.

## Prerequisites
- ClickHouse credentials configured in `config.py`

## Outputs
- `data/{YEAR}/eth_tx_value_output/chunks/` - ETH daily chunk files
- `data/{YEAR}/eth_tx_value_output/weekly/` - ETH weekly aggregated files
- `data/{YEAR}/erc20_tx_value_output/chunks/` - ERC20 daily chunk files
- `data/{YEAR}/erc20_tx_value_output/weekly/` - ERC20 weekly aggregated files

## Setup

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.path.dirname(''), 'functions'))

from eth_data_fetcher import download_eth_transactions, aggregate_to_weekly
from erc20_data_fetcher import download_erc20_transfers, aggregate_to_weekly as aggregate_erc20_weekly
import credentials

print('Imports OK ✓')

## Configuration

In [ ]:
# ══════════════════════════════════════════════════════════════
#  CONFIGURATION  —  edit this cell
# ══════════════════════════════════════════════════════════════

YEAR       = 2020
CHUNK_SIZE = 1000   # number of blocks per query

# ── Derived (do not edit) ──────────────────────────────────────
START_DATE       = f'{YEAR}-01-01'
END_DATE         = f'{YEAR}-12-31'
ETH_OUTPUT_DIR   = f'./data/{YEAR}/eth_tx_value_output'
ERC20_OUTPUT_DIR = f'./data/{YEAR}/erc20_tx_value_output'

print(f'Year       : {YEAR}')
print(f'Date range : {START_DATE} → {END_DATE}')
print(f'ETH output : {ETH_OUTPUT_DIR}')
print(f'ERC20 out  : {ERC20_OUTPUT_DIR}')
print(f'Chunk size : {CHUNK_SIZE} blocks')
print('Configuration set ✓')

## Part 1: Download ETH Native Transactions

Downloads in block-range chunks to avoid timeouts.  
Each chunk is saved immediately. Re-running resumes from where it left off.

In [ ]:
download_eth_transactions(
    start_date=START_DATE,
    end_date=END_DATE,
    output_dir=ETH_OUTPUT_DIR,
    clickhouse_host=credentials.CLICKHOUSE_HOST,
    clickhouse_user=credentials.CLICKHOUSE_USER,
    clickhouse_password=credentials.CLICKHOUSE_PASSWORD,
    chunk_size=CHUNK_SIZE,
)

### Aggregate ETH to Weekly Files

In [ ]:
aggregate_to_weekly(
    chunks_dir=f"{ETH_OUTPUT_DIR}/chunks",
    weekly_dir=f"{ETH_OUTPUT_DIR}/weekly",
)

## Part 2: Download ERC20 Token Transfers

Downloads ERC20 transfers in the same chunked, resumable format.

In [ ]:
download_erc20_transfers(
    start_date=START_DATE,
    end_date=END_DATE,
    output_dir=ERC20_OUTPUT_DIR,
    clickhouse_host=credentials.CLICKHOUSE_HOST,
    clickhouse_user=credentials.CLICKHOUSE_USER,
    clickhouse_password=credentials.CLICKHOUSE_PASSWORD,
    chunk_size=CHUNK_SIZE,
)

### Aggregate ERC20 to Weekly Files

In [ ]:
aggregate_erc20_weekly(
    chunks_dir=f'{ERC20_OUTPUT_DIR}/chunks',
    weekly_dir=f'{ERC20_OUTPUT_DIR}/weekly',
)

## Done!

You should now have:
- ETH transaction data in `data/{YEAR}/eth_tx_value_output/weekly/`
- ERC20 transfer data in `data/{YEAR}/erc20_tx_value_output/weekly/`

Next step: Run notebook 2 (ranking) to generate node rankings.